# PII Detection Guardrail

**What is PII?** Personally Identifiable Information: data that can identify a person, such as an email address, phone number, name or card number.

**How this guardrail helps:** it checks text before it moves on, for example a user's prompt before it's sent to an LLM, or an LLM's answer before it's shown to a user. PII can then be masked or blocked instead of leaking to a third-party model, logs or another user. The [Detect PII](https://guardrailsai.com/hub/validator/guardrails/detect_pii) validator runs locally using Microsoft Presidio.

**One-time setup** (already done in `myenv`, the kernel for this notebook):
```bash
pip install guardrails-ai guardrails-ai-detect-pii
python -m guardrails_ai.detect_pii.post_install   # downloads the NLTK data and spaCy en_core_web_lg (about 400 MB)
```

In [1]:
import logging
import warnings

# silence harmless warnings from spaCy/torch, Presidio's non-English recognizers and guardrails' sync fallback in Jupyter
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="Could not obtain an event loop")
logging.getLogger("presidio-analyzer").setLevel(logging.ERROR)

from dotenv import load_dotenv
from guardrails import Guard  ## the main wrapper for Guardrails
from guardrails.errors import ValidationError
from guardrails_ai.detect_pii import DetectPII

# load the env vars
load_dotenv()

True

## 1. Create the validator and the guard

`on_fail="fix"` replaces any PII it finds with a placeholder such as `<EMAIL_ADDRESS>`.

`Guard().use(...)` needs a validator **instance**. The form shown in the docs, `Guard().use(DetectPII, [...], "exception")`, raises `AttributeError: ... no attribute 'on_fail_descriptor'`.

In [2]:
# create the validator
validator = DetectPII(pii_entities=["EMAIL_ADDRESS", "PHONE_NUMBER"], on_fail="fix")  # mask the PII if the validator fails

# create guard and add the validator to the guard
guard = Guard().use(validator)
guard.configure(allow_metrics_collection=False)

## 2. Validate user input

In [3]:
user_input_1 = "Write an official mail to the RTO office regarding some issues in my driving license"
user_input_2 = "Write an official mail to the RTO office regarding some issues in my driving license. My email is amitkr123@gmail.com and my phone number is 8920333365."

response_1 = guard.validate(user_input_1)

print(f"Validation Result: {response_1.validation_passed}")
print(f"Raw User Input: {response_1.raw_llm_output}")
print(f"Validated Output: {response_1.validated_output}")

Validation Result: True
Raw User Input: Write an official mail to the RTO office regarding some issues in my driving license
Validated Output: Write an official mail to the RTO office regarding some issues in my driving license


With `on_fail="fix"`, `validation_passed` stays `True` even when PII was found, because the text was fixed. To find out whether PII was present, check `validation_summaries`.

In [4]:
response_2 = guard.validate(user_input_2)

print(f"Validation Result: {response_2.validation_passed}")
print(f"PII Found: {len(response_2.validation_summaries) > 0}")
print(f"Raw User Input: {response_2.raw_llm_output}")
print(f"Validated Output: {response_2.validated_output}")

Validation Result: True
PII Found: True
Raw User Input: Write an official mail to the RTO office regarding some issues in my driving license. My email is amitkr123@gmail.com and my phone number is 8920333365.
Validated Output: Write an official mail to the RTO office regarding some issues in my driving license. My email is <EMAIL_ADDRESS> and my phone number is <PHONE_NUMBER>.


## 3. `on_fail="exception"` — block instead of masking

The error message contains the original text, including the PII, so don't write it to logs as it is.

In [5]:
exception_guard = Guard().use(DetectPII(pii_entities=["EMAIL_ADDRESS", "PHONE_NUMBER"], on_fail="exception"))
exception_guard.configure(allow_metrics_collection=False)

try:
    exception_guard.validate(user_input_2)
except ValidationError:
    print("Blocked: the input contains PII")

Blocked: the input contains PII


## 4. Choose which entities to detect

`pii_entities` accepts a list of [Presidio entity types](https://microsoft.github.io/presidio/supported_entities/) or one of two presets:

- `"pii"`: `EMAIL_ADDRESS`, `PHONE_NUMBER`, `DOMAIN_NAME`, `IP_ADDRESS`, `DATE_TIME`, `LOCATION`, `PERSON`, `URL`
- `"spi"` (sensitive data): `CREDIT_CARD`, `CRYPTO`, `IBAN_CODE`, `NRP`, `MEDICAL_LICENSE`, `US_BANK_NUMBER`, `US_DRIVER_LICENSE`, `US_ITIN`, `US_PASSPORT`, `US_SSN`

You can also override the entities for a single call using `metadata`.

In [6]:
text = "Jane Doe lives in Boston. Email jane.doe@example.com"

print(guard.validate(text).validated_output)  # only the email and phone number from the validator
print(guard.validate(text, metadata={"pii_entities": ["PERSON"]}).validated_output)
print(guard.validate(text, metadata={"pii_entities": "pii"}).validated_output)

Jane Doe lives in Boston. Email <EMAIL_ADDRESS>
<PERSON> lives in Boston. Email jane.doe@example.com
<PERSON> lives in <LOCATION>. Email <EMAIL_ADDRESS>


In [7]:
spi_guard = Guard().use(DetectPII(pii_entities="spi", on_fail="fix"))
spi_guard.configure(allow_metrics_collection=False)

print(spi_guard.validate("My card is 4111 1111 1111 1111.").validated_output)
print(spi_guard.validate("My SSN is 536-22-8726.").validated_output)

My card is <CREDIT_CARD>.
My SSN is <US_SSN>.


## 5. What it can miss

Detection relies on patterns and a language model, so check it against the kinds of data your users actually enter:

- **Only the listed entities are checked:** a name passes unless `PERSON` is included.
- **Disguised values can pass:** an email written as "jane dot doe at example dot com" isn't detected.
- **The `spi` preset is US-focused:** the Aadhaar-style number below isn't caught.
- **Labels can be wrong:** a passport number below is masked, but labelled `<US_DRIVER_LICENSE>`.

In [8]:
print(guard.validate("My name is Jane Doe.").validated_output)
print(guard.validate("My email is jane dot doe at example dot com").validated_output)
print(spi_guard.validate("My Aadhaar is 2345 6789 0123.").validated_output)
print(spi_guard.validate("My passport number is P1234567.").validated_output)

My name is Jane Doe.
My email is jane dot doe at example dot com
My Aadhaar is 2345 6789 0123.
My passport number is <US_DRIVER_LICENSE>.
